## 2026 EY AI & 데이터 챌린지 - Landsat 데이터 추출 노트북

이 노트북은 Landsat 데이터 추출 및 벤치마크 노트북에서 사용할 출력 파일 생성을 시연합니다. 기준 데이터는 MS Planetary Computer 카탈로그의 [Landsat Collection 2 Level 2](https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2) 데이터입니다.

<b>주의</b> ... 이 노트북은 Landsat 아카이브에서 데이터 추출에 사용되는 9319개의 데이터 포인트(고유한 위치 및 시간)가 있으므로 상당한 실행 시간이 필요합니다. 코드는 일반적인 노트북 컴퓨터와 일반적인 인터넷 연결에서 완료하는 데 약 7시간이 걸립니다. 데이터 추출 프로세스를 최적화하고 클라우드 컴퓨팅 서비스를 사용하면 실행 시간을 단축할 수 있습니다.

# 터미널이나 노트북 셀에서 실행
pip install pystac_client planetary_computer odc-stac --break-system-packages

In [1]:
pip install pystac_client planetary_computer odc-stac --break-system-packages

Note: you may need to restart the kernel to use updated packages.


### 파이썬 의존성 로드

In [2]:
import warnings
warnings.filterwarnings("ignore")

# 데이터 조작 및 분석
import numpy as np
import pandas as pd

# STAC API 액세스 및 인증을 위한 Planetary Computer 도구
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os

<h3>API 호출을 사용한 Landsat 데이터 추출</h3> <p align="justify"> API 기반 방법을 사용하면 특정 좌표 및 기간에 대한 <b>Landsat</b> 데이터에 효율적으로 액세스하여 프로세스의 확장성과 재현성을 보장할 수 있습니다. </p> <p align="justify"> API를 통해 개별 밴드를 쿼리하거나 즉석에서 <b>NDMI</b>와 같은 지수를 계산할 수 있습니다. 이 접근 방식은 저장소 요구 사항을 줄이고 데이터 전처리 과정을 단순화하여 대규모 환경 및 수질 분석에 이상적입니다. </p>

<p><b>compute_Landsat_values</b> 함수는 각 지점 주위에 100m 초점 버퍼를 사용하여 특정 샘플링 위치에 대한 Landsat 표면 반사율 값을 추출합니다. 각 위치에 대해:</p>

<ul>
  <li>위도 및 경도 좌표 주위에 경계 상자(bbox)가 생성됩니다.</li>
  <li>날짜 범위 내에서 Landsat-8 레벨-2 표면 반사율 이미지에 대해 Microsoft Planetary Computer API를 쿼리합니다.</li>
  <li>가장 가까운 저운량(<10% 구름량) 장면이 선택되고 지정된 밴드(<b>green</b>, <b>nir08</b>, <b>swir16</b>, <b>swir22</b>)가 로드됩니다.</li>
  <li>노이즈 또는 이상값의 영향을 줄이기 위해 경계 상자 내 픽셀의 중앙값이 계산됩니다.</li>
</ul>

<p><b>버퍼 값이 0.00089831인 이유:</b></p>

<p>각 지점 주위에 약 100m의 버퍼를 원합니다. 적도에서 1도 ≈ 110km입니다. 따라서 100m에 해당하는 도는 다음과 같습니다:</p>

<p style="text-align:center;">
  <em>버퍼_도 = 100m / 110,000m/도 ≈ 0.00089831</em>
</p>

<p>이 약간 조정된 값은 버퍼가 Landsat 이미지의 픽셀 해상도와 거의 일치하도록 하여 각 샘플링 위치 주위에 약 100m 영역을 캡처합니다.</p>


In [ ]:
# 설정
tqdm.pandas()

def compute_Landsat_values(row):
    lat = row['Latitude']
    lon = row['Longitude']
    date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')

    # ~100m 버퍼 크기
    bbox_size = 0.00089831  
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2
    ]

    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )

    # 더 넓은 검색 범위, 나중에 가장 가까운 날짜로 필터링
    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime="2011-01-01/2015-12-31",
        query={"eo:cloud_cover": {"lt": 10}},
    )
    
    items = search.item_collection()

    if not items:
        return pd.Series({
            "nir": np.nan, "green": np.nan, "swir16": np.nan, "swir22": np.nan
        })

    try:
        # 샘플 날짜를 UTC로 변환
        sample_date_utc = date.tz_localize("UTC") if date.tzinfo is None else date.tz_convert("UTC")

        # 샘플 날짜에 가장 가까운 항목 선택
        items = sorted(
            items,
            key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date_utc)
        )
        selected_item = pc.sign(items[0])

        # 필요한 밴드 로드
        bands_of_interest = ["green", "nir08", "swir16", "swir22"]
        data = stac_load([selected_item], bands=bands_of_interest, bbox=bbox).isel(time=0)

        green = data["green"].astype("float")
        nir = data["nir08"].astype("float")
        swir16 = data["swir16"].astype("float")
        swir22 = data["swir22"].astype("float")
        
        # 중앙값 계산
        median_green = float(green.median(skipna=True).values)
        median_nir = float(nir.median(skipna=True).values)
        median_swir16 = float(swir16.median(skipna=True).values)
        median_swir22 = float(swir22.median(skipna=True).values)

        # 0을 NaN으로 바꾸기
        median_green = median_green if median_green != 0 else np.nan
        median_nir = median_nir if median_nir != 0 else np.nan
        median_swir16 = median_swir16 if median_swir16 != 0 else np.nan
        median_swir22 = median_swir22 if median_swir22 != 0 else np.nan
        
        return pd.Series({
            "nir": median_nir,
            "green": median_green,
            "swir16": median_swir16,
            "swir22": median_swir22,
        })
    
    except Exception as e:
        return pd.Series({
            "nir": np.nan, "green": np.nan, "swir16": np.nan, "swir22": np.nan
        })


### 훈련 데이터셋의 특징 추출

In [ ]:
Water_Quality_df=pd.read_csv('water_quality_training_dataset.csv')
Water_Quality_df.head()

In [ ]:
Water_Quality_df.shape

In [ ]:
Water_Quality_df_200 = Water_Quality_df.loc[0:199]
Water_Quality_df_200.shape

<h3>참고:</h3>
<p>9,319개 모든 위치에 대한 Landsat 데이터 추출 프로세스는 단일 실행 시 일반적으로 7시간 이상 소요됩니다. 장시간 실행하는 동안 간혹 API 제한, 시간 초과 오류 또는 요청 실패가 발생할 수 있습니다. 이러한 중단을 방지하려면 더 작은 배치로 추출을 실행하는 것이 좋습니다.</p>

<p>이 노트북에서는 처음 200개 위치에 대한 데이터를 추출하는 방법을 보여주는 샘플 코드 스니펫을 제공합니다. 참가자는 동일한 배치 접근 방식을 따라 9,319개 모든 위치에 대한 데이터를 안전하고 효율적으로 추출하는 것이 좋습니다.</p>

<p>저희는 이미 9,319개 모든 위치에 대한 전체 추출을 실행하고 출력을 <b>landsat_features_training.csv</b>에 저장했으며, 이는 벤치마크 노트북에서 사용될 것입니다.
마찬가지로 참가자는 배치로 Landsat 특징을 추출하고, 배치 출력을 결합한 다음, 최종 병합된 데이터셋을 <b>landsat_features_training.csv</b>로 저장하여 벤치마크 노트북이 원활하게 실행되도록 할 수 있습니다.</p>

In [ ]:
# 훈련 데이터셋에 대한 Landsat에서 밴드 값 추출
train_features_path = "landsat_features_training_200.csv"

print("🚀 훈련 데이터에 대한 Landsat 특징 추출 실행 중...")
landsat_train_features = Water_Quality_df_200.progress_apply(compute_Landsat_values, axis=1)
landsat_train_features.to_csv(train_features_path, index=False)

<p><b>NDMI 및 MNDWI 지수:</b></p>
<p>이 노트북에서는 추출된 Landsat 밴드에서 일반적으로 사용되는 두 가지 수질 관련 지수를 계산합니다:</p>
<ul>
  <li><b>NDMI (정규화 차이 수분 지수):</b> 식생 수분 함량 및 표면 수분을 측정합니다. <em>(NIR - SWIR16) / (NIR + SWIR16)</em>으로 계산됩니다.</li>
  <li><b>MNDWI (수정된 정규화 차이 수질 지수):</b> 수질 반사율을 향상시키고 건물 밀집 지역을 억제하여 개방 수역 특징을 강조합니다. <em>(Green - SWIR16) / (Green + SWIR16)</em>으로 계산됩니다.</li>
</ul>

<p>0으로 나누는 것을 방지하기 위해 분모에 <b>엡실론 값</b>(<em>eps = 1e-10</em>)이 추가됩니다. 이러한 지수는 수문학 및 수질 분석에서 물의 존재 및 식생 수분 수준을 감지하는 데 널리 사용됩니다.</p>


In [ ]:
# 지수 생성: NDMI 및 MNDWI
eps = 1e-10
landsat_train_features['NDMI'] = (landsat_train_features['nir'] - landsat_train_features['swir16']) / (landsat_train_features['nir'] + landsat_train_features['swir16'] + eps)
landsat_train_features['MNDWI'] = (landsat_train_features['green'] - landsat_train_features['swir16']) / (landsat_train_features['green'] + landsat_train_features['swir16'] + eps)

In [ ]:
landsat_train_features['Latitude'] = Water_Quality_df['Latitude']
landsat_train_features['Longitude'] = Water_Quality_df['Longitude']
landsat_train_features['Sample Date'] = Water_Quality_df['Sample Date']
landsat_train_features = landsat_train_features[['Latitude', 'Longitude', 'Sample Date', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI']]

In [ ]:
landsat_train_features.to_csv(train_features_path, index=False)

In [ ]:
# 파일 미리보기
landsat_train_features.head()

### 검증 데이터셋의 특징 추출

In [ ]:
Validation_df=pd.read_csv('submission_template.csv')
Validation_df.head()

In [ ]:
Validation_df.shape

In [ ]:
# 제출 데이터셋에 대한 Landsat에서 밴드 값 추출
val_features_path = "landsat_features_validation.csv"

print("🚀 검증 데이터에 대한 Landsat 특징 추출 실행 중...")
landsat_val_features = Validation_df.progress_apply(compute_Landsat_values, axis=1)
landsat_val_features.to_csv(val_features_path, index=False)

In [ ]:
# 지수 생성: NDMI 및 MNDWI
eps = 1e-10
landsat_val_features['NDMI'] = (landsat_val_features['nir'] - landsat_val_features['swir16']) / (landsat_val_features['nir'] + landsat_val_features['swir16'])
landsat_val_features['MNDWI'] = (landsat_val_features['green'] - landsat_val_features['swir16']) / (landsat_val_features['green'] + landsat_val_features['swir16'] + eps)

In [ ]:
landsat_val_features['Latitude'] = Validation_df['Latitude']
landsat_val_features['Longitude'] = Validation_df['Longitude']
landsat_val_features['Sample Date'] = Validation_df['Sample Date']
landsat_val_features = landsat_val_features[['Latitude', 'Longitude', 'Sample Date', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI']]

In [ ]:
landsat_val_features.to_csv(val_features_path, index=False)

In [ ]:
# 파일 미리보기
landsat_val_features.head()